In [1]:
import os

In [2]:
os.chdir("../")

In [4]:
import pandas as pd

data = pd.read_csv("artifacts/data_ingestion/winequality-red.csv")

In [6]:
data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [7]:
data.isnull().sum()

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [9]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataValidationConfig:
    root_dir : Path
    STATUS_FILE : str
    unzip_data_dir : Path
    all_schema : dict

In [10]:
from src.datascience.config import *
from src.datascience.utils.common import read_yaml, create_directories, load_json, save_json

In [12]:
class ConfiurationManager:
    def __init__(self, config_file_path = CONFIG_FILE_PATH, 
                 params_file_path = PARAMS_FILE_PATH,
                 schema_file_path = SCHEMA_FILE_PATH):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifacts_root,], True)
        
    

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir = config.root_dir,
            STATUS_FILE = config.STATUS_FILE,
            unzip_data_dir = config.unzip_data_dir,
            all_schema = self.schema.COLUMNS
        )

        return data_validation_config

In [ ]:
import urllib.request as request
from src.datascience import logger
import zipfile
import os

In [23]:

class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config
    
    def validate_all_columns(self) -> bool:
        try:
            validation_status = True
            data = pd.read_csv(self.config.unzip_data_dir)
            
            for column in self.config.all_schema.keys():
                if column not in data.columns:
                    with open(self.config.STATUS_FILE, "a") as f:
                        f.write(f"Column {column} is not present in the data")
                    validation_status = False

            if validation_status:
                with open(self.config.STATUS_FILE, "a") as f:
                    f.write("All columns are present in the data")
            return validation_status
        
        except Exception as e:
            logger.exception(e)
            raise e

In [24]:
config = ConfiurationManager()
data_validation_config = config.get_data_validation_config()
data_validation = DataValidation(config=data_validation_config)
data_validation.validate_all_columns()

[2026-03-29 17:22:16,206: INFO: common]: created directory at: artifacts
[2026-03-29 17:22:16,207: INFO: common]: created directory at: artifacts/data_validation


True